## Example Input

In [ ]:
import cv2
import numpy as np
import tempfile
import os
from datasets import load_dataset
from IPython.display import Video, display

print("Initializing streams...")
landmark_stream = load_dataset("bdanko/how2sign-landmarks-front-raw-parquet", split="train", streaming=True)
rgb_stream = load_dataset("bdanko/how2sign-rgb-front-clips", split="train", streaming=True)

landmark_sample = next(iter(landmark_stream))
target_id = landmark_sample['video_id']
landmarks = np.frombuffer(landmark_sample['features'], dtype=np.float32).reshape(landmark_sample['shape'])

print(f"Searching for RGB clip: {target_id}...")
rgb_sample = None
for sample in rgb_stream:
    if sample['__key__'] == target_id:
        rgb_sample = sample
        break

if rgb_sample is None:
    print(f"Error: Could not find RGB clip for {target_id}")
else:
    video_data = rgb_sample['mp4']
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tf:
        if isinstance(video_data, bytes): tf.write(video_data)
        elif isinstance(video_data, dict) and 'bytes' in video_data: tf.write(video_data['bytes'])
        else: # Handle path
            tf.close()
            os.remove(tf.name)
            tf.name = video_data if isinstance(video_data, str) else video_data['path']
        video_path = tf.name

    cap = cv2.VideoCapture(video_path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    
    output_path = 'overlay_result.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    print(f"Rendering {len(landmarks)} frames to {output_path}...")
    
    for frame_idx in range(len(landmarks)):
        ret, frame = cap.read()
        if not ret: break
        
        curr_lms = landmarks[frame_idx]
        for i, lm in enumerate(curr_lms):
            x, y = int(lm[0] * w), int(lm[1] * h)
            # Pose=Blue, Face=White, Hands=Green
            color = (255, 0, 0) if i < 33 else (255, 255, 255) if i < 501 else (0, 255, 0)
            cv2.circle(frame, (x, y), 2, color, -1)
        
        out.write(frame)
        if frame_idx % 50 == 0:
            print(f"Progress: {frame_idx}/{len(landmarks)} frames...")

    cap.release()
    out.release()
    
    display(Video(output_path, embed=True, width=640))

## Inference

In [ ]:
from huggingface_hub import snapshot_download
import torch
from inference import load_model_for_inference, preprocess_sample, translate

# load model
repo_id = "bdanko/cslt-flan-t5-small-translator"
print(f"Downloading model from HuggingFace: {repo_id}...")
ckpt_dir = snapshot_download(repo_id=repo_id, local_dir="checkpoints/hf_model")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {device}...")
model, tokenizer, config_dict = load_model_for_inference(ckpt_dir, device=device, tag="best")

# inference
# INPUT: LANDMARKS
# OUTPUT: TEXT
features, padding_mask = preprocess_sample(landmarks, device=device)
prediction = translate(model, tokenizer, features, padding_mask)

print("\n" + "="*40)
print(f"Video ID:    {target_id}")
print(f"Reference:   {landmark_sample.get('sentence', 'N/A')}")
print(f"Prediction:  {prediction}")
print("="*40)